# Research Notebook 02: Comparative Analysis of Volatility Estimators
## Historical Close-to-Close vs. High-Low-Open-Close Range Estimators

### 1. Mathematical Framework
- **Close-to-Close (Standard):**
  $$\sigma_{CC}^2 = \frac{1}{N-1} \sum_{i=1}^N (r_i - \bar{r})^2$$
- **Parkinson (1980):**
  $$\sigma_P^2 = \frac{1}{4 \ln 2} \frac{1}{N} \sum_{i=1}^N \left(\ln\frac{H_i}{L_i}\right)^2$$
- **Garman-Klass (1980):**
  $$\sigma_{GK}^2 = \frac{1}{N} \sum_{i=1}^N \left[ 0.5\left(\ln\frac{H_i}{L_i}\right)^2 - (2\ln 2 - 1)\left(\ln\frac{C_i}{O_i}\right)^2 \right]$$
- **Rogers-Satchell (1991):**
  $$\sigma_{RS}^2 = \frac{1}{N} \sum_{i=1}^N \left[ \ln\frac{H_i}{C_i}\ln\frac{H_i}{O_i} + \ln\frac{L_i}{C_i}\ln\frac{L_i}{O_i} \right]$$


In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from volatility_lab.data.providers.synthetic import SyntheticMarketDataProvider
from volatility_lab.data.preprocessing import DataPreprocessor
from volatility_lab.estimators.historical import HistoricalVolatilityEstimator
from volatility_lab.estimators.ewma import EWMAVolatilityEstimator
from volatility_lab.estimators.parkinson import ParkinsonEstimator
from volatility_lab.estimators.garman_klass import GarmanKlassEstimator
from volatility_lab.estimators.rogers_satchell import RogersSatchellEstimator

raw = SyntheticMarketDataProvider(seed=42).fetch_data('NIFTY50', '2020-01-01', '2023-12-31')
df = DataPreprocessor().clean_and_prepare(raw)
returns = df['log_return']

hist_vol = HistoricalVolatilityEstimator(windows=20).estimate(returns)['hist_vol_20d']
ewma_vol = EWMAVolatilityEstimator(decay_factor=0.94).estimate(returns)
park_vol = ParkinsonEstimator(window=20).estimate(df['High'], df['Low'])
gk_vol = GarmanKlassEstimator(window=20).estimate(df['Open'], df['High'], df['Low'], df['Close'])
rs_vol = RogersSatchellEstimator(window=20).estimate(df['Open'], df['High'], df['Low'], df['Close'])

comp_df = pd.DataFrame({
    'Hist (20d)': hist_vol,
    'EWMA (0.94)': ewma_vol,
    'Parkinson': park_vol,
    'Garman-Klass': gk_vol,
    'Rogers-Satchell': rs_vol
}).dropna()


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5), dpi=100)
comp_df.plot(ax=ax, linewidth=1.4)
ax.set_title("Volatility Estimator Trajectories Comparison (Annualized %)")
ax.set_ylabel("Annualized Volatility")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 2. Efficiency Comparison & Conclusions
- Parkinson, Garman-Klass, and Rogers-Satchell exploit intraday price extremes, reducing sample variance by 5x to 8x compared to close-to-close returns.
- Rogers-Satchell handles non-zero market drift without upward variance bias.
